# Weight initialization

In [1]:
import tensorflow as tf
from tensorflow.keras.initializers import HeNormal
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, Flatten, BatchNormalization
from tensorflow.keras.datasets import mnist
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import LearningRateScheduler
from tensorflow.keras.optimizers import Adam

In [2]:
# Definie the model with 'He' initialization
model = Sequential([
    Flatten(input_shape=(28, 28)),
    Dense(units=64, activation='relu', kernel_initializer=HeNormal()),
    Dense(10, activation='softmax')
])

c:\Users\anima\AppData\Local\pypoetry\Cache\virtualenvs\imb-ai-engineering-KgAhWUw3-py3.10\lib\site-packages\keras\src\layers\reshaping\flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


# Learning rate scheduling

In [3]:
(x_train, y_train), (x_val, y_val) = mnist.load_data() 
x_train, x_val = x_train / 255.0, x_val / 255.0 

In [4]:
def scheduler(epoch, lr):
    if epoch < 10:
        return lr
    else:
        return float(lr * tf.math.exp(-0.1))
    
lr_scheduler = LearningRateScheduler(scheduler)

# Model Training

In [5]:
# Compile the model with an optimizer and loss function
model.compile(optimizer=Adam(), loss='sparse_categorical_crossentropy',metrics=['accuracy'])

# Train the model with a learning rate scheduler
history = model.fit(x_train, y_train,
                    validation_data=(x_val, y_val),
                    epochs=20,
                    callbacks=[lr_scheduler])

Epoch 1/20
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.8547 - loss: 0.5117 - val_accuracy: 0.9461 - val_loss: 0.1808 - learning_rate: 0.0010
Epoch 2/20
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 3s 1ms/step - accuracy: 0.9521 - loss: 0.1616 - val_accuracy: 0.9587 - val_loss: 0.1328 - learning_rate: 0.0010
Epoch 3/20
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 3s 1ms/step - accuracy: 0.9655 - loss: 0.1170 - val_accuracy: 0.9672 - val_loss: 0.1080 - learning_rate: 0.0010
Epoch 4/20
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 3s 1ms/step - accuracy: 0.9741 - loss: 0.0881 - val_accuracy: 0.9688 - val_loss: 0.0967 - learning_rate: 0.0010
Epoch 5/20
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 3s 1ms/step - accuracy: 0.9778 - loss: 0.0719 - val_accuracy: 0.9711 - val_loss: 0.0912 - learning_rate: 0.0010
Epoch 6/20
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 3s 1ms/step - accuracy: 0.9818 - loss: 0.0586 - val_accuracy: 0.9717 - val_loss: 0.0897 - learning_rate: 0.0010
Epoch 7/20
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 3s 1ms/step - accuracy: 0.9858 - l

# Batch normalization

Normalizes the input layer by adjusting and scaling the activations

In [6]:
model = Sequential([
    Flatten(input_shape=(28, 28)),
    Dense(units=128, activation='relu'),
    BatchNormalization(),
    Dense(10, activation='softmax')
])

# Mixed precision training

Users both 16-bit and 32-bit floating-point types t ospeed up training on modern GPUs

In [7]:
from tensorflow.keras import mixed_precision

# Enable mixed precision
policy = mixed_precision.Policy('mixed_float16')
mixed_precision.set_global_policy(policy)

# Model pruning

Reduces the number of parameters in a model by removing less significant connections or neurons

In [10]:
#Example Code for Model Pruning:
import tensorflow_model_optimization as tfmot
prune_low_magnitude = tfmot.sparsity.keras.prune_low_magnitude

# Apply pruning to the model
pruning_params = {'pruning_schedule': 
tfmot.sparsity.keras.PolynomialDecay(initial_sparsity=0.0,
final_sparsity=0.5,
begin_step=0,
end_step=2000)}

model_pruned = prune_low_magnitude(model, **pruning_params)

ValueError: `prune_low_magnitude` can only prune an object of the following types: keras.models.Sequential, keras functional model, keras.layers.Layer, list of keras.layers.Layer. You passed an object of type: Sequential.

# Quantization
Reduces the precision of the numbers used to represent the model's weights.

In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
quantized_model = converter.convert()

# Save the quantized model
with open('quantized_model_tflite', 'wb') as f:
    f.write(quantized_model)